In [3]:
# ==========================================================
# Physics-Informed Deep Unfolding (PIDU) Framework
# for Bearing Fault Diagnosis Under Variable Speed/Load
# (Implements Sections 3.1-3.6 of the manuscript)
# ==========================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

# ==========================================================
# SECTION 3.3: PHYSICS-GUIDED FAULT-FREQUENCY PRIOR (Eq. 8-10)
# ==========================================================
def fault_frequencies(fr, Nb=9, db=7.94e-3, Dp=39e-3, phi=0.0):
    ratio = (db / Dp) * np.cos(phi)
    fo = (Nb / 2) * fr * (1 - ratio)
    fi = (Nb / 2) * fr * (1 + ratio)
    fb = (Dp / (2 * db)) * fr * (1 - ratio ** 2)
    return fo, fi, fb

# ==========================================================
# SECTION 4.1: SYNTHETIC VIBRATION DATA GENERATION (Eq. 1)
# ==========================================================
def generate_signal(label, fs=2560, N=2048, fr=20.0):
    t = np.arange(N) / fs
    fo, fi, fb = fault_frequencies(fr)
    signal = np.zeros(N)
    if label == 0:  # Healthy Bearing (HB)
        signal += 0.6*np.sin(2*np.pi*600*t) + 0.4*np.sin(2*np.pi*950*t)
    else:
        f_fault = {1: fo, 2: fi, 3: fb}[label]
        period = fs / f_fault
        impact_times = np.arange(0, N, period)
        decay = {1: 22, 2: 30, 3: 55}[label]
        carrier = {1: 1400, 2: 1700, 3: 2200}[label]   # distinct resonance carrier per fault type
        amp = {1: 3.2, 2: 2.8, 3: 2.4}[label]
        for it in impact_times:
            tau = t - it/fs
            mask = tau >= 0
            signal[mask] += amp*np.exp(-decay*tau[mask])*np.sin(2*np.pi*carrier*tau[mask])
        if label == 2:
            # inner race: strong once-per-rev load-zone amplitude modulation
            signal *= (1 + 0.5*np.sin(2*np.pi*fr*t))
        if label == 3:
            # ball fault: double-impact (inner+outer contact) + slip jitter
            period2 = period/2.0
            impact_times2 = np.arange(0, N, period2)
            for it in impact_times2:
                tau = t - it/fs
                mask = tau >= 0
                signal[mask] += 0.6*amp*np.exp(-decay*tau[mask])*np.sin(2*np.pi*carrier*tau[mask])
            jitter = np.random.normal(0, 0.08, N)
            signal = signal*(1+jitter)
    noise = np.random.normal(0, 0.10, N)
    x = (signal+noise).astype(np.float32)
    return x, np.array([fo, fi, fb], dtype=np.float32)

def build_dataset(n_per_class=250, fs=2560, N=2048):
    X,y,FREQS=[],[],[]
    speeds_hz=[15.0,20.0,25.0,30.0]
    for label in range(4):
        for _ in range(n_per_class):
            fr=np.random.choice(speeds_hz)
            x,freqs=generate_signal(label,fs=fs,N=N,fr=fr)
            X.append(x); y.append(label); FREQS.append(freqs)
    return np.stack(X), np.array(y), np.stack(FREQS)

# ==========================================================
# SECTION 3.5.2: HILBERT TRANSFORM (FFT-based, for envelope features)
# ==========================================================
def hilbert_transform(x, dim=-1):
    Nlen = x.shape[dim]
    Xf = torch.fft.fft(x, dim=dim)
    h = torch.zeros(Nlen, device=x.device)
    if Nlen % 2 == 0:
        h[0]=1; h[Nlen//2]=1; h[1:Nlen//2]=2
    else:
        h[0]=1; h[1:(Nlen+1)//2]=2
    shape=[1]*x.dim(); shape[dim]=Nlen
    h=h.view(shape)
    return torch.fft.ifft(Xf*h, dim=dim)

# ==========================================================
# SECTION 3.1-3.6: PIDU NETWORK
# ==========================================================
class PIDU(nn.Module):
    def __init__(self, L=5, K=4, N=2048, fs=2560, M=16, num_classes=4, n_subbands=8, beta_init=2.0):
        super().__init__()
        self.L,self.K,self.N,self.fs,self.M=L,K,N,fs,M
        self.Nf=N//2+1
        self.register_buffer('freq_bins', torch.fft.rfftfreq(N, d=1/fs))
        self.alpha_raw = nn.Parameter(torch.tensor(0.0))
        self.tau_raw = nn.Parameter(torch.tensor(0.0))
        beta_raw_init = np.log(np.exp(beta_init)-1) if beta_init>0 else 0.0
        self.beta_raw = nn.Parameter(torch.tensor(float(beta_raw_init)))
        self.b_km = nn.Parameter(torch.zeros(K,3))
        self.n_subbands=n_subbands
        self.mask_params = nn.Parameter(torch.zeros(K, n_subbands))
        band_idx = torch.linspace(0, n_subbands-1e-6, self.Nf).long()
        self.register_buffer('band_idx', band_idx)

        feat_dim = K*M + K  # RMS features + per-mode log-energy feature
        self.bn_in = nn.BatchNorm1d(feat_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.BatchNorm1d(64),
            nn.Linear(64, num_classes)
        )

    def alpha(self): return F.softplus(self.alpha_raw)+1e-3
    def tau(self):   return F.softplus(self.tau_raw)+1e-3
    def beta(self):  return F.softplus(self.beta_raw)+1e-3

    def forward(self, x, f_targets):
        B = x.shape[0]
        x_hat = torch.fft.rfft(x, dim=-1)                                  # FFT, Eq. 2
        u_hat = torch.zeros(B, self.K, self.Nf, dtype=torch.complex64, device=x.device)
        lam_hat = torch.zeros(B, self.Nf, dtype=torch.complex64, device=x.device)
        omega = torch.linspace(0, self.fs/2, self.K, device=x.device).unsqueeze(0).repeat(B,1)
        a_km = F.softmax(self.b_km, dim=1)                                 # Eq. 12

        for _ in range(self.L):
            # ---- mode (primal) update, Eq. 14 ----
            sum_all = u_hat.sum(dim=1, keepdim=True)
            sum_others = sum_all - u_hat
            numerator = x_hat.unsqueeze(1) - sum_others + lam_hat.unsqueeze(1)/2
            freq_diff = self.freq_bins.view(1,1,-1) - omega.unsqueeze(-1)
            denom = (1 + 2*self.alpha()*freq_diff**2).to(torch.complex64)
            u_hat = numerator/denom

            # ---- physics-guided center-frequency update, Eq. 15 ----
            power = u_hat.abs()**2
            A = (power*self.freq_bins.view(1,1,-1)).sum(dim=-1)
            Bk = power.sum(dim=-1)+1e-8
            prior_num = torch.einsum('km,bm->bk', a_km, f_targets)
            prior_den = a_km.sum(dim=1).unsqueeze(0)
            omega = (A + self.beta()*prior_num) / (Bk + self.beta()*prior_den)

            # ---- dual-variable update, Eq. 16 ----
            lam_hat = lam_hat + self.tau()*(x_hat - u_hat.sum(dim=1))

        # ---- adaptive spectral mask, applied to final-stage modes (Eq. 17-18) ----
        r = self.mask_params[:, self.band_idx]
        mask = torch.sigmoid(r).unsqueeze(0)
        u_tilde_hat = mask*u_hat
        u_tilde = torch.fft.irfft(u_tilde_hat, n=self.N, dim=-1)

        # ---- Hilbert envelope (Eq. 19-20) ----
        analytic = hilbert_transform(u_tilde, dim=-1)
        envelope = analytic.abs()

        # ---- fixed-dimensional RMS feature extraction (Eq. 21) ----
        frame_len = self.N//self.M
        frames = envelope.unfold(-1, frame_len, frame_len)
        rms = torch.sqrt((frames**2).mean(dim=-1)+1e-8)     # [B,K,M]
        rms_feat = rms.reshape(B,-1)

        # ---- extra per-mode spectral energy feature (new in v2) ----
        mode_energy = torch.log1p(power.sum(dim=-1))         # [B,K]
        feat = torch.cat([rms_feat, mode_energy], dim=1)
        feat = self.bn_in(feat)

        # ---- classification head (Eq. 22-23) ----
        logits = self.classifier(feat)

        # ---- reconstruction signal for Lrec, Eq. 25 ----
        recon = u_tilde.sum(dim=1)
        return logits, recon

# ==========================================================
# SECTION 3.6: TOTAL LOSS (Eq. 24-26)
# ==========================================================
def total_loss(logits,y,recon,x,gamma=1e-3):
    Lcls = nn.CrossEntropyLoss()(logits,y)
    Lrec = F.mse_loss(recon,x)
    return Lcls+gamma*Lrec, Lcls.item(), Lrec.item()

# ==========================================================
# SECTION 4.2: HYPERPARAMETERS (Table 4) AND DATASET BUILD
# ==========================================================
N_SAMPLES=2048; FS=2560; L_STAGES=5; K_MODES=4; M_FRAMES=16
BATCH_SIZE=64; EPOCHS=80; LR=2e-3; WEIGHT_DECAY=1e-4; GAMMA=1e-3

X,y,FREQS = build_dataset(n_per_class=250, fs=FS, N=N_SAMPLES)

# ---- z-score normalization (Eq. 29) ----
mu,sigma = X.mean(axis=1,keepdims=True), X.std(axis=1,keepdims=True)+1e-8
X=(X-mu)/sigma

# ---- train/validation/test split, 70/10/20 (Table 2) ----
n_total=len(y)
perm=np.random.permutation(n_total)
X,y,FREQS = X[perm], y[perm], FREQS[perm]
train_end=int(0.70*n_total); val_end=int(0.80*n_total)
X_train,y_train,F_train = X[:train_end], y[:train_end], FREQS[:train_end]
X_val,y_val,F_val = X[train_end:val_end], y[train_end:val_end], FREQS[train_end:val_end]
X_test,y_test,F_test = X[val_end:], y[val_end:], FREQS[val_end:]

train_X=torch.tensor(X_train).to(device); train_y=torch.tensor(y_train).to(device); train_F=torch.tensor(F_train).to(device)
val_X=torch.tensor(X_val).to(device); val_y=torch.tensor(y_val).to(device); val_F=torch.tensor(F_val).to(device)
test_X=torch.tensor(X_test).to(device); test_y=torch.tensor(y_test).to(device); test_F=torch.tensor(F_test).to(device)

# ==========================================================
# MODEL, OPTIMIZER, LR SCHEDULE (Table 4: Adam(W) + cosine annealing)
# ==========================================================
model = PIDU(L=L_STAGES,K=K_MODES,N=N_SAMPLES,fs=FS,M=M_FRAMES,num_classes=4).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ==========================================================
# TRAINING LOOP (with validation + early stopping, Sec. 4.2)
# ==========================================================
best_val_loss=float('inf'); best_state=None; patience=15; bad_epochs=0

for epoch in range(EPOCHS):
    model.train()
    perm_idx = torch.randperm(len(train_y))
    train_loss, train_correct = 0.0, 0
    for i in range(0, len(perm_idx), BATCH_SIZE):
        idx = perm_idx[i:i+BATCH_SIZE]
        xb,yb,fb_ = train_X[idx], train_y[idx], train_F[idx]
        optimizer.zero_grad()
        logits, recon = model(xb, fb_)
        loss,lcls,lrec = total_loss(logits,yb,recon,xb,gamma=GAMMA)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # stabilizes the complex-FFT path
        optimizer.step()
        train_loss += loss.item()*len(idx)
        train_correct += (logits.argmax(1)==yb).sum().item()
    train_loss/=len(train_y); train_acc=train_correct/len(train_y)
    scheduler.step()

    # ---- validation ----
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(val_y), BATCH_SIZE):
            xb,yb,fb_ = val_X[i:i+BATCH_SIZE], val_y[i:i+BATCH_SIZE], val_F[i:i+BATCH_SIZE]
            logits, recon = model(xb, fb_)
            loss,_,_ = total_loss(logits,yb,recon,xb,gamma=GAMMA)
            val_loss += loss.item()*len(yb)
            val_correct += (logits.argmax(1)==yb).sum().item()
    val_loss/=len(val_y); val_acc=val_correct/len(val_y)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss=val_loss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad_epochs=0
    else:
        bad_epochs+=1

    if (epoch+1)%5==0 or epoch==0:
        print(f"Epoch {epoch+1:02d}: Train Loss={train_loss:.4f} Acc={train_acc:.3f} | "
              f"Val Loss={val_loss:.4f} Acc={val_acc:.3f}  beta={model.beta().item():.2f}")

    if bad_epochs>=patience:
        print(f"Early stopping at epoch {epoch+1}")
        break
if best_state is not None:
    model.load_state_dict(best_state)

# =============================
# TESTING /  ACCURACY
# =============================
model.eval()
all_logits, all_labels = [],[]
with torch.no_grad():
    for i in range(0, len(test_y), BATCH_SIZE):
        xb,yb,fb_ = test_X[i:i+BATCH_SIZE], test_y[i:i+BATCH_SIZE], test_F[i:i+BATCH_SIZE]
        logits,_ = model(xb, fb_)
        all_logits.append(logits.cpu()); all_labels.append(yb.cpu())
all_logits=torch.cat(all_logits,0); all_labels=torch.cat(all_labels,0)
probs=F.softmax(all_logits,dim=1).numpy(); preds=probs.argmax(1); labels_np=all_labels.numpy()

acc=accuracy_score(labels_np,preds)
prec,rec,f1,_ = precision_recall_fscore_support(labels_np,preds,average='macro',zero_division=0)
try:
    auc=roc_auc_score(labels_np,probs,multi_class='ovr')
except ValueError:
    auc=float('nan')
cm=confusion_matrix(labels_np,preds)
print("\nFinal Test Results:")
print(f"Accuracy:  {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall:    {rec*100:.2f}%")
print(f"F1-score:  {f1*100:.2f}%")
print(f"AUC:       {auc:.3f}")
print('Results on Synthetic Data to understand network performace:')
print(cm)

Epoch 01: Train Loss=1.2690 Acc=0.436 | Val Loss=1.2988 Acc=0.360  beta=1.99
Epoch 05: Train Loss=0.6974 Acc=0.776 | Val Loss=0.7701 Acc=0.790  beta=1.98
Epoch 10: Train Loss=0.5067 Acc=0.821 | Val Loss=0.4463 Acc=0.890  beta=1.96
Epoch 15: Train Loss=0.5303 Acc=0.836 | Val Loss=0.4371 Acc=0.820  beta=1.95
Epoch 20: Train Loss=0.3917 Acc=0.871 | Val Loss=0.3678 Acc=0.890  beta=1.95
Epoch 25: Train Loss=0.3623 Acc=0.890 | Val Loss=0.3930 Acc=0.860  beta=1.95
Epoch 30: Train Loss=0.3749 Acc=0.871 | Val Loss=0.2845 Acc=0.920  beta=1.95
Epoch 35: Train Loss=0.3189 Acc=0.900 | Val Loss=0.2920 Acc=0.910  beta=1.94
Epoch 40: Train Loss=0.4244 Acc=0.843 | Val Loss=0.3913 Acc=0.830  beta=1.94
Epoch 45: Train Loss=0.2910 Acc=0.910 | Val Loss=0.2103 Acc=0.960  beta=1.94
Epoch 50: Train Loss=0.2612 Acc=0.910 | Val Loss=0.2518 Acc=0.910  beta=1.94
Epoch 55: Train Loss=0.2956 Acc=0.909 | Val Loss=0.2119 Acc=0.940  beta=1.94
Early stopping at epoch 59

Final Test Results:
Accuracy:  90.50%
Precision: